In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import math
from functools import partial

__all__ = [
    'ResNet', 'resnet10', 'resnet18', 'resnet34', 'resnet50', 'resnet101',
    'resnet152', 'resnet200'
]


def conv3x3x3(in_planes, out_planes, stride=1, dilation=1):
    # 3x3x3 convolution with padding
    return nn.Conv3d(
        in_planes,
        out_planes,
        kernel_size=3,
        dilation=dilation,
        stride=stride,
        padding=dilation,
        bias=False)


def downsample_basic_block(x, planes, stride, no_cuda=False):
    out = F.avg_pool3d(x, kernel_size=1, stride=stride)
    zero_pads = torch.Tensor(
        out.size(0), planes - out.size(1), out.size(2), out.size(3),
        out.size(4)).zero_()
    if not no_cuda:
        if isinstance(out.data, torch.cuda.FloatTensor):
            zero_pads = zero_pads.cuda()

    out = Variable(torch.cat([out.data, zero_pads], dim=1))

    return out


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, dilation=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3x3(inplanes, planes, stride=stride, dilation=dilation)
        self.bn1 = nn.BatchNorm3d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3x3(planes, planes, dilation=dilation)
        self.bn2 = nn.BatchNorm3d(planes)
        self.downsample = downsample
        self.stride = stride
        self.dilation = dilation

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, inplanes, planes, stride=1, dilation=1, downsample=None):
        super(Bottleneck, self).__init__()
        self.conv1 = nn.Conv3d(inplanes, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(
            planes, planes, kernel_size=3, stride=stride, dilation=dilation, padding=dilation, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.conv3 = nn.Conv3d(planes, planes * 4, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride
        self.dilation = dilation

    def forward(self, x):
        residual = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out


class ResNet(nn.Module):

    def __init__(self,
                 block,
                 layers,
                 sample_input_D,
                 sample_input_H,
                 sample_input_W,
                 num_seg_classes,
                 shortcut_type='B',
                 no_cuda = False):
        self.inplanes = 64
        self.no_cuda = no_cuda
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv3d(
            1,
            64,
            kernel_size=7,
            stride=(2, 2, 2),
            padding=(3, 3, 3),
            bias=False)
            
        self.bn1 = nn.BatchNorm3d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(3, 3, 3), stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0], shortcut_type)
        self.layer2 = self._make_layer(
            block, 128, layers[1], shortcut_type, stride=2)
        self.layer3 = self._make_layer(
            block, 256, layers[2], shortcut_type, stride=1, dilation=2)
        self.layer4 = self._make_layer(
            block, 512, layers[3], shortcut_type, stride=1, dilation=4)
        
        self.batch_norm = nn.BatchNorm3d(2048)
        
        self.global_average_pooling = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 4),
            nn.Softmax(dim = -1)
        )
        
#         self.conv_seg = nn.Sequential(
#                                         nn.ConvTranspose3d(
#                                         512 * block.expansion,
#                                         32,
#                                         2,
#                                         stride=2
#                                         ),
#                                         nn.BatchNorm3d(32),
#                                         nn.ReLU(inplace=True),
#                                         nn.Conv3d(
#                                         32,
#                                         32,
#                                         kernel_size=3,
#                                         stride=(1, 1, 1),
#                                         padding=(1, 1, 1),
#                                         bias=False), 
#                                         nn.BatchNorm3d(32),
#                                         nn.ReLU(inplace=True),
#                                         nn.Conv3d(
#                                         32,
#                                         num_seg_classes,
#                                         kernel_size=1,
#                                         stride=(1, 1, 1),
#                                         bias=False) 
#                                         )

        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                m.weight = nn.init.kaiming_normal(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, shortcut_type, stride=1, dilation=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            if shortcut_type == 'A':
                downsample = partial(
                    downsample_basic_block,
                    planes=planes * block.expansion,
                    stride=stride,
                    no_cuda=self.no_cuda)
            else:
                downsample = nn.Sequential(
                    nn.Conv3d(
                        self.inplanes,
                        planes * block.expansion,
                        kernel_size=1,
                        stride=stride,
                        bias=False), nn.BatchNorm3d(planes * block.expansion))

        layers = []
        layers.append(block(self.inplanes, planes, stride=stride, dilation=dilation, downsample=downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes, dilation=dilation))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.batch_norm(x)
        x = self.global_average_pooling(x)
        x = self.flatten(x)
        x = self.fc(x)

        return x

def resnet10(**kwargs):
    """Constructs a ResNet-18 model.
    """
    model = ResNet(BasicBlock, [1, 1, 1, 1], **kwargs)
    return model


def resnet18(**kwargs):
    """Constructs a ResNet-18 model.
    """
    model = ResNet(BasicBlock, [2, 2, 2, 2], **kwargs)
    return model


def resnet34(**kwargs):
    """Constructs a ResNet-34 model.
    """
    model = ResNet(BasicBlock, [3, 4, 6, 3], **kwargs)
    return model


def ResNet50(**kwargs):
    """Constructs a ResNet-50 model.
    """
    model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
    return model


In [ ]:
from monai.transforms import LoadImage, Randomizable, apply_transform, Transpose, Rotate90
from monai.transforms import AddChannel, Compose, RandRotate90, Resize, ScaleIntensity, ToTensor, RandAffine, Rotate
from monai.utils import get_seed

depth, height, width = 40, 64, 64
rotation_angle = 90

train_transform = Compose([
    LoadImage(image_only=True),
    Transpose((2, 0, 1)),
    AddChannel(),
    ScaleIntensity(), 
    Resize((None, height, width)), 
    RandAffine( 
        prob=0.5,
        translate_range=(1, 5, 5),
        #rotate_range=(0, np.pi * 2, np.pi * 2),
        scale_range=(0.15, 0.15, 0.15),
        padding_mode='border'
    ),
    Rotate90(spatial_axes = (1, 2)),
    ToTensor()])

val_transform = Compose([LoadImage(image_only=True), Transpose((2, 0, 1)), AddChannel(), ScaleIntensity(), 
                         Resize((None, height, width)), Rotate90(spatial_axes = (1, 2)), ToTensor()])
test_transform = Compose([LoadImage(image_only=True), Transpose((2, 0, 1)), AddChannel(), ScaleIntensity(), 
                          Resize((None, height, width)), Rotate90(spatial_axes = (1, 2)), ToTensor()])

In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets

# train_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(30),
#     #transforms.RandomVerticalFlip(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

# val_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

# test_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

def contrast_stretch(image, new_min=0, new_max=1):
    # Compute the current min and max intensity values
    current_min = np.min(image)
    current_max = np.max(image)
    
    # Apply contrast stretching
    stretched_image = (image - current_min) * (new_max - new_min) / (current_max - current_min) + new_min
    
    return stretched_image

def z_threshold(image):
    mean = np.mean(image)
    std = np.std(image)
    
    image_copy = image.copy()
    z_norm = (image - mean) / std
    
    for i in range(len(z_norm)):
        for j in range(len(z_norm[i])):
            if (z_norm[i][j] >= 4):
                image_copy[i][j] = 0
    return image_copy

def load_and_preprocess_images(file_path, transform):
    image_data = nib.load(file_path).get_fdata().transpose(2, 0, 1)
    
#     preprocessed_data = np.array([contrast_stretch(np.rot90(np.expand_dims(img, axis = -1), 
#                                            k = 1)) for img in image_data])[10:-10]
    
#     augmented_data = []
#     for i in preprocessed_data:
#         transformed = transform((i * 255).astype(np.uint8))
#         print(np.shape(transformed))
#         #print(np.shape(i))
#         augmented_data.append(transformed)
#     augmented_data = torch.stack(augmented_data).permute(1, 0, 2, 3)
    
    # turn transforms to monai transforms
    augmented_data = transform(file_path)[:, 10:-10]
    print(augmented_data.shape)
    #augmented_data = np.array(augmented_data)
    #print(np.shape(augmented_data))
    #print(s[:-7])
    return augmented_data
    #return augmented_data

import nibabel as nib

directory = 'Splitted'

x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []

used_patients_train = []
used_patients_val = []
used_patients_test = []


for subset in os.listdir(directory):
    if (subset != ".DS_Store"):
        print(subset)
        subset_path = os.path.join(directory, subset)
        for diagnosis in os.listdir(subset_path):
            if (diagnosis != ".DS_Store"):
                diagnosis_path = os.path.join(subset_path, diagnosis)
                for filename in os.listdir(diagnosis_path):
                    print(diagnosis)
                    if (filename != ".DS_Store"):
                        file_path = os.path.join(diagnosis_path, filename)

                        if (subset == 'train'):
                            x_train.append(load_and_preprocess_images(file_path, train_transform))

                            if (diagnosis == 'CN'):
                                y_train.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_train.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_train.append(2)
                            else:
                                y_train.append(3)
                            
                            used_patients_train.append(filename)

                        elif (subset == 'val'):
                            x_val.append(load_and_preprocess_images(file_path, val_transform))

                            if (diagnosis == 'CN'):
                                y_val.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_val.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_val.append(2)
                            else:
                                y_val.append(3)
                            
                            used_patients_val.append(filename)
                        else:
                            x_test.append(load_and_preprocess_images(file_path, test_transform))

                            if (diagnosis == 'CN'):
                                y_test.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_test.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_test.append(2)
                            else:
                                y_test.append(3)
                            
                            used_patients_test.append(filename)

In [ ]:
def get_weighted_random_sample(label_list):
    class_frequencies = np.bincount(label_list)
    print(class_frequencies)
    # Calculate the inverse of class frequencies as probabilities
    class_probabilities = 1.0 / class_frequencies
    #print(class_probabilities)
    # Normalize the probabilities to sum to 1
    class_probabilities /= np.sum(class_probabilities)
    #print(self.label_list)
    return class_probabilities[label_list]
train_weights = get_weighted_random_sample(y_train)
val_weights = get_weighted_random_sample(y_val)
test_weights = get_weighted_random_sample(y_test)

In [ ]:
weighted_train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)
weighted_val_sampler = WeightedRandomSampler(
    weights = val_weights,
    num_samples = len(val_weights),
    replacement = True
)
weighted_test_sampler = WeightedRandomSampler(
    weights = test_weights,
    num_samples = len(test_weights),
    replacement = True
)

In [ ]:
x_train, x_val, x_test = torch.stack(x_train), torch.stack(x_val), torch.stack(x_test)
y_train, y_val, y_test = torch.tensor(y_train), torch.tensor(y_val), torch.tensor(y_test)

In [ ]:
plt.figure()
plt.imshow(x_train[20][:, 10].permute(1, 2, 0))
plt.show()
# ncdhw

In [ ]:
from torch.utils.data import TensorDataset
batch_size = 32

train_set = TensorDataset(x_train, y_train)
val_set = TensorDataset(x_val, y_val)
test_set = TensorDataset(x_test, y_test)
train_loader = DataLoader(train_set, batch_size = batch_size, sampler = weighted_train_sampler)
val_loader = DataLoader(train_set, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_set, batch_size = batch_size, shuffle = True)

In [ ]:
print(y_val)

In [ ]:
for x, y in train_loader:
    print(y)

In [ ]:
val_weights

In [ ]:
import monai
#from monai.networks.nets import resnet18, resnet34, resnet50

device = torch.device("cpu")
model = ResNet50(sample_input_D = 20, sample_input_H = 128, sample_input_W = 128, num_seg_classes = 4)
PATH_PRETRAINED_WEIGHTS = "MedicalNet/MedicalNet_pytorch_files2/pretrain/resnet_50_23dataset.pth"
weights_dict = torch.load(PATH_PRETRAINED_WEIGHTS, device)
weights_dict = {k.replace('module.', ''): v for k, v in weights_dict['state_dict'].items()}
model_dict = model.state_dict()
model_dict.update(weights_dict)
model.load_state_dict(model_dict)



In [ ]:
from torchsummary import summary
summary(model, (1, 20, 128, 128))

In [ ]:
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
criterion = CrossEntropyLoss()
num_epochs = 15
optimizer = torch.optim.SGD(model.parameters(), lr= 0.001) 
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, val_loader, model, criterion, optimizer)

In [ ]:
test_loop(test_loader, model, criterion)